## Imports

In [1]:
import openai
import os 
import json

from qdrant_client import QdrantClient
from qdrant_client.models import Filter, FieldCondition, MatchValue



## Download Qdrant Data

In [2]:
qdrant_client = QdrantClient(url="http://localhost:6333")

/Users/jooaobrum/Library/CloudStorage/GoogleDrive-joao.paulo.brum14@gmail.com/My Drive/Projetos Pessoais/Projetos de Estudo/end2end-ai-engineering-bootcamp/hephaestus-agentic-maintenance/.venv/lib/python3.12/site-packages/qdrant_client/qdrant_remote.py:280: UserWarning: Qdrant client version 1.17.1 is incompatible with server version 1.12.0. Major versions should match and minor version difference must not exceed 1. Set check_compatibility=False to skip version check.
  show_warning(


In [15]:
collection_name = "cm_interventions"

all_points = qdrant_client.scroll(
    collection_name=collection_name,
    limit=100,
    offset=None,
    with_payload=True,
    with_vectors=False
)[0]

In [ ]:
import re

def extract_fault_code(summary):
    m = re.search(r'\[FAULT_CODE\]\s*(\S+)', summary)
    return m.group(1) if m else None

all_contexts = [
    {
        'id':         p.payload['id'],
        'machine':    p.payload.get('machine', 'N/A'),
        'date':       p.payload.get('date_start', 'N/A'),
        'summary':    p.payload.get('summary', 'N/A'),
        'fault_code': extract_fault_code(p.payload.get('summary', '')),
    }
    for p in all_points
]

## Running a Prompt to Generate Eval Dataset

In [ ]:
import random
import json
import pandas as pd
from collections import defaultdict

STYLES = [
    "Very objective and brief (e.g., 'RF issue', 'No pressure')",
    "Direct and declarative (e.g., 'Machine tripping during cycle')",
    "Natural conversational question (e.g., 'The machine won't start — what should I check first?')",
    "Full technical inquiry (e.g., 'Following a low pressure alarm, please list the checkpoints.')",
]

PROMPT_SINGLE = """\
You are a Maintenance Engineer creating a high-quality evaluation dataset for a RAG-based AI assistant.

Given the maintenance intervention record below, generate one realistic question a technician would ask
and its ideal answer based solely on the record.

### Intervention Record:
{records}

### Instructions:
- **Question style**: {style}
- The question must reflect only what is described in the record (symptom / fault observed).
- Do NOT mention "ID", "machine", "record", "context", or "provided data" in the question.
- Mentioning the fault code in the question is optional.
- The answer must state: (1) most likely root cause, (2) main action(s) that solved the issue.
- chunks_id must contain only the ID of the record above.

### Output (strict JSON, no extra text):
{{
  "question": "...",
  "answer": "...",
  "reasoning": "...",
  "chunks_id": ["{first_id}"]
}}
"""

PROMPT_MULTI = """\
You are a Maintenance Engineer creating a high-quality evaluation dataset for a RAG-based AI assistant.

You are given {n} maintenance intervention records for the same machine that share the same fault code ({fault_code}).
Generate ONE realistic question a technician or supervisor would ask that can only be answered
by consulting ALL of these records together (e.g., recurring fault pattern, root cause variation across dates,
comparison of repair effectiveness).

### Intervention Records:
{records}

### Instructions:
- **Question style**: {style}
- The question must require information from more than one record to be answered properly.
- Do NOT mention "IDs", "records", "context", or "provided data" in the question.
- Mentioning the fault code in the question is optional.
- The answer must synthesise findings across the records: common root causes, differences, recommended actions.
- chunks_id must list ALL record IDs provided above.

### Output (strict JSON, no extra text):
{{
  "question": "...",
  "answer": "...",
  "reasoning": "...",
  "chunks_id": {id_list}
}}
"""

def format_records(contexts):
    parts = []
    for ctx in contexts:
        parts.append(
            f"ID: {ctx['id']}\nMachine: {ctx['machine']}\nDate: {ctx['date']}\nSummary:\n{ctx['summary']}"
        )
    return "\n\n---\n\n".join(parts)

def context_text(contexts):
    return "\n\n".join(
        f"Machine: {c['machine']}\nDate: {c['date']}\nSummary: {c['summary']}"
        for c in contexts
    )

# --- Group by (machine, fault_code) — skip records with no fault code ---
by_machine_fault = defaultdict(list)
for ctx in all_contexts:
    if ctx["fault_code"]:
        by_machine_fault[(ctx["machine"], ctx["fault_code"])].append(ctx)

# Build 2–3 record groups from each (machine, fault_code) bucket with ≥2 records
multi_groups = []
for (machine, fault_code), records in by_machine_fault.items():
    if len(records) < 2:
        continue
    random.shuffle(records)
    for i in range(0, len(records) - 1, 2):
        size = random.choice([2, 3]) if i + 2 < len(records) else 2
        multi_groups.append((fault_code, records[i : i + size]))

random.shuffle(multi_groups)

# --- Generate samples ---
synthetic_data = []

# Single-context samples (one per record)
for ctx in all_contexts:
    style = random.choice(STYLES)
    prompt = PROMPT_SINGLE.format(
        records=format_records([ctx]),
        style=style,
        first_id=ctx["id"],
    )
    try:
        response = openai.chat.completions.create(
            model="gpt-5.4-nano",
            messages=[{"role": "user", "content": prompt}],
            response_format={"type": "json_object"},
        )
        g = json.loads(response.choices[0].message.content)
        synthetic_data.append({
            "sample_type":  "single",
            "context_ids":  [ctx["id"]],
            "machine":      ctx["machine"],
            "fault_code":   ctx["fault_code"],
            "question":     g["question"],
            "ground_truth": g["answer"],
            "reasoning":    g.get("reasoning", ""),
            "chunks_id":    g.get("chunks_id", [ctx["id"]]),
            "style_used":   style,
            "context_text": context_text([ctx]),
        })
        print(f"✓ single  {ctx['id']} | {ctx['machine']} | {ctx['fault_code']}")
    except Exception as e:
        print(f"✗ single  {ctx['id']}: {e}")

# Multi-context samples
for fault_code, group in multi_groups:
    style = random.choice(STYLES)
    ids = [c["id"] for c in group]
    prompt = PROMPT_MULTI.format(
        n=len(group),
        fault_code=fault_code,
        records=format_records(group),
        style=style,
        id_list=json.dumps(ids),
    )
    try:
        response = openai.chat.completions.create(
            model="gpt-5.4-nano",
            messages=[{"role": "user", "content": prompt}],
            response_format={"type": "json_object"},
        )
        g = json.loads(response.choices[0].message.content)
        synthetic_data.append({
            "sample_type":  "multi",
            "context_ids":  ids,
            "machine":      group[0]["machine"],
            "fault_code":   fault_code,
            "question":     g["question"],
            "ground_truth": g["answer"],
            "reasoning":    g.get("reasoning", ""),
            "chunks_id":    g.get("chunks_id", ids),
            "style_used":   style,
            "context_text": context_text(group),
        })
        print(f"✓ multi   {ids} | {group[0]['machine']} | {fault_code}")
    except Exception as e:
        print(f"✗ multi   {ids}: {e}")

df_eval = pd.DataFrame(synthetic_data)
df_eval.to_csv("synthetic_eval_dataset.csv", index=False)
print(f"\nDone! {len(synthetic_data)} rows ({df_eval['sample_type'].value_counts().to_dict()}) saved to synthetic_eval_dataset.csv")
df_eval.head()

In [25]:
df_eval.iloc[0]

context_id                                            INT-2022-0569
question          We received a maintenance request INT-2022-056...
ground_truth      INT-2022-0569 has no machine model/site asset ...
reasoning         Because the record provides only an interventi...
chunks_id                                           [INT-2022-0569]
context_source        ID: INT-2022-0569\nMachine: N/A\nSummary: N/A
Name: 0, dtype: object

## Upload to LangSmith

In [ ]:
import dotenv
from langsmith import Client

dotenv.load_dotenv()

DATASET_NAME = "hephaestus-rag-eval"

ls_client = Client()

# Create dataset (or get existing)
if ls_client.has_dataset(dataset_name=DATASET_NAME):
    dataset = ls_client.read_dataset(dataset_name=DATASET_NAME)
    print(f"Using existing dataset: {dataset.id}")
else:
    dataset = ls_client.create_dataset(
        dataset_name=DATASET_NAME,
        description="Synthetic RAG evaluation dataset for maintenance interventions. "
                    "Contains single-context and multi-context (same machine + fault code) samples.",
    )
    print(f"Created dataset: {dataset.id}")

# Build examples — inputs/outputs follow LangSmith convention
examples = [
    {
        "inputs":  {
            "question":     row["question"],
            "context_text": row["context_text"],
        },
        "outputs": {
            "ground_truth": row["ground_truth"],
            "reasoning":    row["reasoning"],
            "chunks_id":    row["chunks_id"],
        },
        "metadata": {
            "sample_type": row["sample_type"],
            "machine":     row["machine"],
            "fault_code":  row.get("fault_code"),
            "style_used":  row["style_used"],
            "context_ids": row["context_ids"],
        },
    }
    for _, row in df_eval.iterrows()
]

ls_client.create_examples(dataset_id=dataset.id, examples=examples)

print(f"\nUploaded {len(examples)} examples to '{DATASET_NAME}'")
print(f"single: {df_eval['sample_type'].eq('single').sum()}  |  multi: {df_eval['sample_type'].eq('multi').sum()}")